In [ ]:
import json
import pandas as pd
import hashlib
import glob
import os

# Ensure the ADOMD.NET path is in the system path prior to importing Pyadomd
from sys import path
adomd_path = r'C:\Program Files\Microsoft.NET\ADOMD.NET\160'
if adomd_path not in path:
    path.append(adomd_path)

# Import Pyadomd after ensuring the path is set
from pyadomd import Pyadomd

def find_visual_id(id_map, start_id):
    current_id = start_id
    while current_id:
        node = id_map.get(current_id)
        if not node:
            break
        metrics = node.get('metrics', {})
        visual_id = metrics.get('visualId')
        if visual_id:
            return visual_id
        current_id = node.get('parentId')
    return None

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

def row_hash(row):
    # Concatenate all values as string and hash
    row_str = '|'.join(str(val) for val in row.values)
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()

baseline_folder = os.path.join(os.getcwd(), "baseline")
#baseline_file = os.path.join(baseline_folder, "baseline.parquet")
baseline_file = os.path.join(os.getcwd(), "baseline.parquet")

if os.path.isdir(baseline_folder):
    if os.path.isfile(baseline_file):
        baseline_exists = True
        print("Baseline folder and baseline.parquet file exist.")
    else:
        baseline_exists = False
        print("Baseline folder exists, but baseline.parquet file is missing.")
elif os.path.isfile(baseline_file):
    baseline_exists = True
    print("Baseline.parquet file exists, but baseline folder is missing.")
else:
    baseline_exists = False

print(f"Baseline {baseline_exists}")

user_value = input("Press enter to create a baseline, otherwise enter a instance name: ")
print(f"You entered: {user_value}")

if user_value.strip() == "":
    instance_name = "baseline"

    if baseline_exists:
        overwrite = input("Do you want to overwrite the baseline (Y/N)? ").strip().lower() 
        if overwrite != 'y':
            print("Exiting without creating a new baseline.")
            exit()
else:
    if not baseline_exists:
        print("Baseline folder does not exist. Please create a baseline first.")
        exit()
    else:
        instance_name = user_value.strip()

# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:52253;Initial Catalog=0089fece-2176-4d08-9a8f-2272c4cfe691"

# Specify the folder containing your JSON files
json_folder = r"C:\Users\Cory.Cundy\source\repos\Power BI Regression Testing"

all_dfs = []


for json_path in glob.glob(os.path.join(json_folder, "*.json")):
    with open(json_path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)
    events = data.get("events", [])
    id_map = {event['id']: event for event in events if 'id' in event}
    flat_events = [flatten_event(e) for e in events]
    df = pd.DataFrame(flat_events)
    if not df.empty:
        df['visualId'] = df['id'].apply(lambda x: find_visual_id(id_map, x))
        all_dfs.append(df)

# Combine all DataFrames into one
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()





# json_path = "Sales & Returns Sample v201912.json"

# with open(json_path, "r", encoding="utf-8-sig") as f:
#     data = json.load(f)

# # events = data["events"]
# events = data.get("events", [])

# # Build a lookup dictionary for fast id -> node access
# id_map = {event['id']: event for event in events if 'id' in event}


# flat_events = [flatten_event(e) for e in data.get("events", [])]

# df = pd.DataFrame(flat_events)

# # Apply to each row in the DataFrame to get a new column with the visualId
# df['visualId'] = df['id'].apply(lambda x: find_visual_id(id_map, x))

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = final_df[final_df['name'] == 'Execute DAX Query'].copy()
filtered_df.drop_duplicates(inplace=True)
filtered_df['result_sets'] = 0
filtered_df['combined_query_hash'] = ""

# filtered_df = df[
#     (df['name'] == 'Execute DAX Query') &
#     (df['id'] == 'baa3553b-c0fa-4f0e-87d6-98ea260fecba')
# ].copy()
# filtered_df




# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    #Loop over each query from the dataframe
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']

        try:
            with conn.cursor().execute(query_text) as cur:
                result_set_index = 0
                query_hashes = []  # List to collect all query hashes
                has_next = True
                while has_next:
                    result_set_index += 1
                    columns = [col[0] for col in cur.description]
                    result_rows = cur.fetchall()
                    print(f"Result set {result_set_index} has {len(result_rows)} rows")

                    if result_rows:
                        # Create a DataFrame for this query's results
                        result_df = pd.DataFrame(result_rows, columns=columns)

                        # Add a hash column for each row
                        if not result_df.empty:
                            result_df['row_hash'] = result_df.apply(row_hash, axis=1)

                            # Sort by row_hash
                            result_df = result_df.sort_values('row_hash').reset_index(drop=True)
                            # Concatenate all row_hash values and hash them
                            row_hashes = '|'.join(result_df['row_hash'].tolist())
                            combined_row_hash = hashlib.sha256(row_hashes.encode('utf-8')).hexdigest()
                            query_hashes.append(combined_row_hash)

                            print(f"query: {query_id} result set {result_set_index} rows: {len(result_df)} combined hash: {combined_row_hash}")
                        else:
                            print(f"query: {query_id} result set {result_set_index} is empty")

                    has_next = cur.nextresult()

                filtered_df.loc[idx, 'result_sets'] = result_set_index

                if query_hashes:
                    combined = '|'.join(query_hashes)
                    combined_query_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
                    print(f"Query combined hash: {combined_query_hash}")

                    # Store in filtered_df (for the first result set only, or adjust as needed)
                    filtered_df.loc[idx, 'combined_query_hash'] = combined_query_hash
                else:
                    filtered_df.loc[idx, 'combined_query_hash'] = None
                    print(f"No result sets for query {query_id}")

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")

# Drop NaN values and convert to string (in case)
all_hashes = filtered_df['combined_query_hash'].dropna().astype(str).tolist()
combined = '|'.join(all_hashes)
final_overall_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
filtered_df['final_overall_hash'] = final_overall_hash

print(f"Final overall hash for all combined_query_hash values: {final_overall_hash}")

filtered_df = filtered_df[['id', 'parentId', 'visualId', 'QueryText', 'RowCount', 'result_sets', 'combined_query_hash', 'final_overall_hash']]

filtered_df.to_csv(f"{instance_name}.csv", index=False)
filtered_df.to_parquet(f"{instance_name}.parquet", index=False)
# df = pd.read_parquet("filtered_df.parquet")

#If this run is not a baseline, then compare to the baseline
if baseline_exists and instance_name != "baseline":
    if not os.path.isfile(baseline_file):
        print("Baseline file does not exist. Please create a baseline first.")
        exit()

    baseline_df = pd.read_parquet(baseline_file)

    # Compare the current filtered_df with the baseline_df
    comparison_df = filtered_df.merge(baseline_df, on='id', suffixes=('', '_baseline'), how='outer', indicator=True)

    # Identify changes
    changes = comparison_df[comparison_df['_merge'] != 'both']
    if not changes.empty:
        print("Changes detected compared to the baseline:")
        print(changes)
    else:
        print("No changes detected compared to the baseline.")


    # Identify value differences in columns for rows present in both
    cols_to_compare = ['combined_query_hash']
    diff_mask = (comparison_df['_merge'] == 'both') & (
        comparison_df[[col for col in cols_to_compare]].ne(
            comparison_df[[f"{col}_baseline" for col in cols_to_compare]].values
        ).any(axis=1)
    )
    value_diffs = comparison_df[diff_mask]
    if not value_diffs.empty:
        print("Rows with differing values between current and baseline:")
        print(value_diffs[['id'] + [col for col in cols_to_compare] + [f"{col}_baseline" for col in cols_to_compare]])
    else:
        print("No value differences in shared rows.")

value_diffs


Baseline.parquet file exists, but baseline folder is missing.
Baseline True
You entered: Test 3
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"XBOX

Result set 1 has 1 rows
query: 2bd3d61f-1ea4-4cb1-aaba-1743fea9a057 result set 1 rows: 1 combined hash: 3ab09c11216d811e73f9c3c93abffb6b9a6523b8b30d98ca4a7f18ea05924403
Result set 2 has 1 rows
query: 2bd3d61f-1ea4-4cb1-aaba-1743fea9a057 result set 2 rows: 1 combined hash: f200f58d10ed1589daff7eae277547507e1fcb0c4bfe44a5ea68119654f16ceb
Query combined hash: 9a156eeccbb5a87944d014fa86a419ee0bf045fc9512ad880061400a88731779
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"XBOX

Result set 1 has 1 rows
query: b1f1c889-6489-4d11-9306-2478268354f5 result set 1 rows: 1 combined hash: 3ab09c11216d811e73f9c3c93abffb6b9a6523b8b30d98ca4a7f18ea05924403
Query combined hash: f8019579cd75ad594237aeb13af961485b302053d900725f9b8a1492ca0ed086
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"XBOX

Result set 1 has 1 rows
query: be5a6d64-78

,id,parentId,visualId,QueryText,RowCount,result_sets,combined_query_hash,final_overall_hash,parentId_baseline,visualId_baseline,QueryText_baseline,RowCount_baseline,result_sets_baseline,combined_query_hash_baseline,final_overall_hash_baseline,_merge
12,3116d681-710e-4c4a-b34b-aa867a01d347,5466294c-f42e-4373-9c22-0dc27418be93,b33397810d555ca70a8c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,16.0,2,a8db92e37a449349fc27882113c44b965f332dad142efb...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,5466294c-f42e-4373-9c22-0dc27418be93,b33397810d555ca70a8c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,16.0,2,113805e845ab8d302bf3a430f549c710267e1bf81c3578...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
24,80d1289d-9fde-4a2a-83f7-7a1e6abfd856,69f38537-2166-447c-b254-f865c96bc6ff,805719ca6000cb000be2,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,27.0,1,1e7f3d5bd1fae11ff21363539c2c3fd4bc979836134ece...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,69f38537-2166-447c-b254-f865c96bc6ff,805719ca6000cb000be2,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,27.0,1,9d249633ca25d0a143eedc3e988b0242502ca1d6e1b021...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
39,c8f382ac-e545-4879-bd35-67f20096b8d7,8e712962-ac31-485e-8c04-c0d576955b18,948988db07a4db09d58d,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,26.0,1,9bcaf8e5411523b21dd4e096d41eee979d58c5d171c28b...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,8e712962-ac31-485e-8c04-c0d576955b18,948988db07a4db09d58d,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,26.0,1,0fea39528cc2f3bfc39d83d0a11d892a376f194ade4cf4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
46,e033aaf7-a2aa-413f-8a2f-969a4deb4615,d96a4290-a4a6-4b4e-9ba8-7da7a725cb34,e5b7b6b90231b41809a1,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,1,4eaca420673e3994a1748aa6bae951e616ad9d4a91861e...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,d96a4290-a4a6-4b4e-9ba8-7da7a725cb34,e5b7b6b90231b41809a1,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,1,a3e30f2f2b65e66d1016efa882c59db7b1b8cf50503758...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
